In [ ]:
using Libdl
using LinearAlgebra
using Plots
using Revise

import ForwardDiff as FD
import MeshCat
import MeshCatMechanisms
import RigidBodyDynamics as RBD
import RotaryInvertedPendulum as RIP

include(joinpath(@__DIR__, "../tinympc-julia/tinympc/TinyMPC.jl"))
using .TinyMPC

In [ ]:
scalar_type = Float64

frequency = 150  # in Hertz
dt = scalar_type(1 / frequency)

x_init = scalar_type[0, π - 0.2, 1, 0]  # initial state
x_goal = scalar_type[0, π, 0, 0]  # goal state

@info "Using $(scalar_type) as the scalar type."

In [ ]:
# Load the URDF model of the rotary inverted pendulum
package_path = joinpath(pkgdir(RIP), "..")
filename = joinpath(package_path, "urdf/model.urdf")
mechanism = RBD.parse_urdf(filename, scalar_type=scalar_type)

In [ ]:
vis = MeshCat.Visualizer()
open(vis)

MeshCat.setprop!(vis["/Cameras/default/rotated/<object>"], "fov", 40)

urdfvisuals = MeshCatMechanisms.URDFVisuals(filename, package_path=[package_path])
mvis = MeshCatMechanisms.MechanismVisualizer(mechanism, urdfvisuals, vis["model"])

# foreach(
#     body_frame -> MeshCatMechanisms.setelement!(mvis, body_frame),
#     RBD.default_frame.(RBD.bodies(mechanism))
# )

In [ ]:
rip_dynamics, A, B = RIP.linearise_dynamics(mechanism, dt)

println("A, state-transition matrix:")
display(A)

println()

println("B, control matrix:")
display(B)

@assert isapprox(A, scalar_type[
 1.0          0.00586205   0.01         1.94951e-5
 6.66429e-19  1.01389     -8.48376e-21  0.0100462
 5.6255e-17   1.17511      1.0          0.00586205
 1.33286e-16  2.78421     -2.54513e-18  1.01389
], atol=1e-3)

@assert isapprox(B, scalar_type[
   0.6367702401730249
   0.6595238918048201
 127.48232693639513
 132.20871156052203
], atol=1e-3)

println()

println("All done.")

In [ ]:
let
    # simulate the dynamics with the zero controller

    duration = scalar_type(4.0)  # in seconds
    num_knots = Int(duration / dt) + 1
    
    x_all = zeros(scalar_type, (4, num_knots))
    u_all = zeros(scalar_type, (1, num_knots))

    x = x_init  # set initial state
    
    for i in 1:num_knots
        x_all[:,i] = x  # save current state
        u = u_all[:,i]  # retrieve controls

        # simulate the current state given the controls
        x = RIP.integrate_rk4(rip_dynamics, [x; u], dt)
    end
    
    RIP.play_trajectory_animation(mvis, x_all[1:2, :], dt)
    RIP.plot_trajectory(x_all[1:2, :], x_all[3:4, :], u_all)
end

In [ ]:
let
    x = x_init  # set initial state
    # x = scalar_type[0, π - 0.1, 0, 0]

    # Riccati recursion on the linearized dynamics
    Q = Diagonal(scalar_type[10, 1, 10, 1])
    R = Diagonal(scalar_type[1])
    P = 1 * Q
    K = zeros(scalar_type, (1, 4))

    for i in 1:1000
        P = Q + A' * P * A - A' * P * B * inv(R + B' * P * B) * B' * P * A
        K = inv(R + B' * P * B) * B' * P * A
    end

    display(K)

    lqr_controller(x::Vector) = -K * x  # The LQR controller
    lqr_controller(rand(scalar_type, 4))

    # simulate the dynamics with the LQR controller

    duration = scalar_type(2.0)  # in seconds
    num_knots = Int(duration / dt) + 1
    
    x_all = zeros(scalar_type, (4, num_knots))
    u_all = zeros(scalar_type, (1, num_knots))
    
    pos_lim = scalar_type(3π/4)   # in radians
    vel_lim = scalar_type(1.0)    # in radians per second
    tau_lim = scalar_type(0.05)  # in Newton meters
    
    for i in 1:num_knots
        x_all[:,i] = x  # save current state

        # x[2] = mod(x[2], 2π)
        u = lqr_controller(x - x_goal)
        # clamp!(u, -tau_lim, tau_lim)
        u_all[:, i] = u  # save LQR command
    
        # simulate the current state given the controls
        x = RIP.integrate_rk4(rip_dynamics, [x; u], dt)
        # @views clamp!(x[1:1], -pos_lim, pos_lim)
        # @views clamp!(x[3:3], -vel_lim, vel_lim)
    end
    
    RIP.play_trajectory_animation(mvis, x_all[1:2, :], dt)
    RIP.plot_trajectory(x_all[1:2, :], x_all[3:4, :], u_all)

end

## Code Generation

In [ ]:
# Your absolute path to the tinympc-Julia directory, you need to change this
path_repo = expanduser("~/git/Rotary Inverted Pendulum")
tinympc_julia_dir = "$(path_repo)/RotaryInvertedPendulum-julia/tinympc-julia"
tinympc_dir = "$(tinympc_julia_dir)/tinympc/TinyMPC"  # Path to the TinyMPC directory (C code)
TinyMPC.compile_lib(tinympc_dir)  # Compile the C code into a shared library

In [ ]:
n = 4  # number of state variables
m = 1  # number of control variables
num_knots_mpc_horizon = 20

Q = Float64[10, 1, 10, 1]
R = Float64[1]
rho = 0.1

# Limits on state and control variables
pos_lim = π/2  # in radians
vel_lim = 5.0   # in radians per second
tau_lim = 0.08  # in Newton meters

# 130 millinewton meter = 0.13 newton meter. Sources:
# https://www.aliexpress.com/item/1005006111249881.html
# https://www.unitconverters.net/moment-of-force/millinewton-meter-to-newton-meter.htm

# box constraints on the state variables
x_min_per_knot = [-pos_lim, -floatmax(Float32), -vel_lim, -floatmax(Float32)]
x_max_per_knot = [ pos_lim,  floatmax(Float32),  vel_lim,  floatmax(Float32)]
x_min = repeat(x_min_per_knot, num_knots_mpc_horizon)
x_max = repeat(x_max_per_knot, num_knots_mpc_horizon)

# box constraints on the control variables
u_min_per_knot = [-tau_lim]
u_max_per_knot = [ tau_lim]
u_min = repeat(u_min_per_knot, num_knots_mpc_horizon - 1)
u_max = repeat(u_max_per_knot, num_knots_mpc_horizon - 1)

abs_prim_tol = 1.0e-2  # absolute primal tolerance
abs_dual_tol = 1.0e-2  # absolute dual tolerance
max_iter = 50          # maximum number of iterations
check_termination = 1  # whether to check termination and period
gen_wrapper = 1        # generate wrapper for high-level interfaces
output_dir = "$(tinympc_julia_dir)/generated_code"  # path to the generated code

# Path to the compiled TinyMPC library. Linux: .so, Mac: .dylib, Windows: .dll
lib_tinympc = "$(tinympc_dir)/build/src/tinympc/libtinympcShared.dylib"

@ccall lib_tinympc.tiny_codegen(
    n::Cint,
    m::Cint,
    num_knots_mpc_horizon::Cint,
    A::Ptr{Float64},
    B::Ptr{Float64},
    Q::Ptr{Float64},
    R::Ptr{Float64},
    x_min::Ptr{Float64},
    x_max::Ptr{Float64},
    u_min::Ptr{Float64},
    u_max::Ptr{Float64},
    rho::Float64,
    abs_prim_tol::Float64,
    abs_dual_tol::Float64,
    max_iter::Cint,
    check_termination::Cint,
    gen_wrapper::Cint,
    tinympc_dir::Ptr{UInt8},
    output_dir::Ptr{UInt8},
)::Cint

TinyMPC.compile_lib(output_dir)

# Path to the compiled library
lib_tinympc_codegen = "$(output_dir)/build/tinympc/libtinympcShared.dylib"
@info lib_tinympc_codegen

In [ ]:
duration = scalar_type(8.0)  # in seconds
num_knots = Int(duration / dt) + 1
@show num_knots

x_all = zeros(4, num_knots)
u_all = zeros(1, num_knots)

# Set the initial state
x = [0, π - 0.2, 0.1, 0]
# x = [0, 0, 0, 0.1]
# x = x_init

# Placeholder for the list of control inputs in the MPC horizon
u = zeros(Float32, (m, num_knots_mpc_horizon - 1))

# Use delta because MPC uses the linearized dynamics around upright position
# Set the reference state to 0 as well as reset
delta_xref = zeros(Float32, (n, num_knots_mpc_horizon))  # reference state
@ccall lib_tinympc_codegen.set_xref(delta_xref::Ptr{Float32}, 0::Cint)::Cvoid

# Reset the dual variables
@ccall lib_tinympc_codegen.reset_dual_variables(0::Cint)::Cvoid

for i in 1:num_knots
    # x[2] = mod(x[2], 2π)

    # if abs(mod(x[2], 2π) - π) < deg2rad(10)
    #     pend_goal = div(x[2], 2π) * 2π

    # Set a new reference state
    if (i == 400)
        new_goal = Float32[1, 0, 0, 0]
        println("Setting new goal to $(new_goal)")
        delta_xref_new = repeat(new_goal, num_knots_mpc_horizon)
        @ccall lib_tinympc_codegen.set_xref(delta_xref_new::Ptr{Float32}, 0::Cint)::Cvoid
    end

    noise = randn(n) * 0.001  # simulate noisy sensors / model error

    # Set initial state from measurement
    delta_x_noise = Vector{Float32}(x + noise - x_goal)
    @ccall lib_tinympc_codegen.set_x0(delta_x_noise::Ptr{Float32}, 0::Cint)::Cvoid  # set the current state
    @ccall lib_tinympc_codegen.call_tiny_solve(0::Cint)::Cvoid                      # solve the problem
    @ccall lib_tinympc_codegen.get_u(u::Ptr{Matrix{Float32}}, 0::Cint)::Cvoid       # get the control input
    clamp!(u, -tau_lim, tau_lim)

    x_all[:,i] = x     # save current state
    u_all[1,i] = u[1]  # save control command

    # Simulate the dynamics
    vars = convert(Vector{Float64}, [x; u[1]])
    x = RIP.integrate_rk4(rip_dynamics, vars, dt)
    @views clamp!(x[1:1], -pos_lim, pos_lim)  # clamp the position of the arm
    x[3] = (x[1] - x_all[1,i]) / dt  # update velocity of the arm after clamp
end

RIP.play_trajectory_animation(mvis, x_all[1:2,:], dt)
RIP.plot_trajectory(x_all[1:2,:], x_all[3:4,:], u_all)

In [ ]:
x_all

In [ ]:
u_all